# Human Brain Segmentation — Track B: BigBrain 9-Class Volume

Fine-tune DINOv2-Large + UperNet on the BigBrain 200um classified volume.
Uses the same model architecture and training recipe as the mouse Run 9 (final)
model, adapted for the BigBrain 3D NIfTI volume with 9 tissue classes.

**Data source:** BigBrain histological volume (696x770x605, 200um isotropic)
with 9-class tissue classification (Merker stain).

**Key differences from mouse training:**
- NIfTI volume via nibabel (not NRRD via pynrrd)
- 10 classes (0=background + 9 tissue classes) vs 1,328 mouse structures
- Gap-based exclusion (gap=2 = 1mm buffer) to mitigate spatial leakage
- Merker stain (different from PhD Nissl tissue)
- Dense annotations (every voxel labeled) — standard CE, no ignore_index needed

## Config (Run 9 recipe)
- DINOv2-Large + UperNet, 10 classes (background + 9 tissue types)
- Last 4 backbone blocks unfrozen (20-23)
- Differential LR: backbone 1e-5, head 1e-4
- Baseline augmentation: flip + rot15 + jitter
- Standard CE loss
- Batch 2 x 2 grad accum = effective batch 4
- 200 epochs
- Interleaved split with gap=2 for spatial leakage mitigation

In [0]:
# Cell 0 — Install project wheel from DBFS

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

Processing /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.1 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.5.2
    Not uninstalling accelerate at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-a7abef9d-7d1a-4b8d-a3e4-3843dc4c19bf
    Can't uninstall 'accelerate'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Cell 1 — Configuration

import os
import mlflow

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-human-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
BIGBRAIN_HISTOLOGY = f"{WORKSPACE_BASE}/bigbrain/histological_volume/full8_200um_optbal.nii.gz"
BIGBRAIN_ANNOTATION = f"{WORKSPACE_BASE}/bigbrain/classified_volume/full_cls_200um_9classes.nii.gz"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Training hyperparameters (Run 9 recipe) ----------
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
NUM_UNFROZEN_BLOCKS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2
CROP_SIZE = 518
NUM_EPOCHS = 200
SPLIT_STRATEGY = "interleaved"
GAP = 2  # 2 slices x 200um = 1mm buffer

OUTPUT_DIR = "/tmp/checkpoints/human-bigbrain"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/human-bigbrain"

HYPERPARAMS = {
    "track": "B (BigBrain 9-class)",
    "crop_size": CROP_SIZE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "num_unfrozen_blocks": NUM_UNFROZEN_BLOCKS,
    "num_epochs": NUM_EPOCHS,
    "freeze_backbone": False,
    "gradient_checkpointing": "backbone_only_non_reentrant",
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "split_strategy": SPLIT_STRATEGY,
    "gap": GAP,
    "model": MODEL_REPO_ID,
    "dataset": "BigBrain 200um 9-class classified volume",
    "augmentation": "baseline: flip_50pct, rot15_always, jitter_always",
    "loss": "CE (dense annotations)",
    "run_description": "Track B: BigBrain 9-class, interleaved split, gap=2, 200ep",
}

print(f"{'='*65}")
print(f"TRACK B: BIGBRAIN 9-CLASS TISSUE SEGMENTATION")
print(f"{'='*65}")
print(f"Volume:          BigBrain 200um (696x770x605)")
print(f"Classes:         10 (background + 9 tissue types)")
print(f"Split:           {SPLIT_STRATEGY} with gap={GAP} (1mm buffer)")
print(f"Batch size:      {BATCH_SIZE} (x{GRADIENT_ACCUMULATION_STEPS} = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Backbone LR:     {BACKBONE_LR}")
print(f"Head LR:         {HEAD_LR}")
print(f"Unfrozen blocks: last {NUM_UNFROZEN_BLOCKS} (blocks {24 - NUM_UNFROZEN_BLOCKS}-23)")
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Output dir:      {FINAL_MODEL_DIR}")

MLflow experiment: /Users/noel.nosse@grainger.com/histology-human-brain-segmentation (ID: 4022263052856815)
TRACK B: BIGBRAIN 9-CLASS TISSUE SEGMENTATION
Volume:          BigBrain 200um (696x770x605)
Classes:         10 (background + 9 tissue types)
Split:           interleaved with gap=2 (1mm buffer)
Batch size:      2 (x2 = effective 4)
Backbone LR:     1e-05
Head LR:         0.0001
Unfrozen blocks: last 4 (blocks 20-23)
Epochs:          200
Output dir:      /dbfs/FileStore/allen_brain_data/models/human-bigbrain


In [0]:
# Cell 2 — Download model weights from JFrog Artifactory mirror

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

(…)496e6d21ea90829a6652ed6f75ec47/README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

(…)6e6d21ea90829a6652ed6f75ec47/config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

(…)d21ea90829a6652ed6f75ec47/.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

(…)a6652ed6f75ec47/preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Model downloaded to: /tmp/dinov2-large/models--facebook--dinov2-large/snapshots/a18992645e496e6d21ea90829a6652ed6f75ec47


In [0]:
# Cell 3 — Build BigBrain datasets
#
# 1. Load NIfTI volumes via BigBrainSlicer
# 2. Build 9-class identity mapping
# 3. Create BrainSegmentationDataset with interleaved split + gap=2

from histological_image_analysis.bigbrain_slicer import BigBrainSlicer
from histological_image_analysis.ontology import (
    build_bigbrain_9class_mapping,
    BIGBRAIN_9CLASS_NAMES,
)
from histological_image_analysis.dataset import BrainSegmentationDataset

# --- Step 1: Load NIfTI volumes ---
slicer = BigBrainSlicer(
    histology_path=BIGBRAIN_HISTOLOGY,
    annotation_path=BIGBRAIN_ANNOTATION,
)
slicer.load_volumes()
print(f"Volume shape: {slicer.image_volume.shape}")
print(f"Num coronal slices: {slicer.num_slices}")

# --- Step 2: Build mapping ---
mapping = build_bigbrain_9class_mapping()
NUM_LABELS = 10  # 0 (background) + 9 tissue classes
class_names = [BIGBRAIN_9CLASS_NAMES.get(i, f"class_{i}") for i in range(NUM_LABELS)]
HYPERPARAMS["num_labels"] = NUM_LABELS
print(f"Mapping: {NUM_LABELS} classes — {class_names}")

# --- Step 3: Show split statistics ---
splits = slicer.get_split_indices(
    split_strategy=SPLIT_STRATEGY, gap=GAP,
)
print(f"Split ({SPLIT_STRATEGY}, gap={GAP}): "
      f"train={len(splits['train'])} | val={len(splits['val'])} | test={len(splits['test'])}")

# --- Step 4: Create datasets ---
train_ds = BrainSegmentationDataset(
    slicer=slicer,
    split="train",
    mapping=mapping,
    crop_size=CROP_SIZE,
    augment=True,
    split_strategy=SPLIT_STRATEGY,
)
val_ds = BrainSegmentationDataset(
    slicer=slicer,
    split="val",
    mapping=mapping,
    crop_size=CROP_SIZE,
    augment=False,
    split_strategy=SPLIT_STRATEGY,
)

HYPERPARAMS["train_samples"] = len(train_ds)
HYPERPARAMS["val_samples"] = len(val_ds)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# Verify a sample
sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")

Volume shape: (696, 770, 605)
Num coronal slices: 696
Mapping: 10 classes — ['Background', 'Gray Matter', 'White Matter', 'Cerebrospinal Fluid', 'Meninges', 'Blood Vessels', 'Bone/Skull', 'Muscle', 'Artifact', 'Other/Unknown']
Split (interleaved, gap=2): train=234 | val=59 | test=59
Train samples: 468 | Val samples: 59
pixel_values: torch.Size([3, 518, 518]), labels: torch.Size([518, 518])


In [0]:
# Cell 4 — Create model with partially unfrozen backbone

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=False,
    pretrained_backbone_path=model_path,
)

first_frozen_block = 24 - NUM_UNFROZEN_BLOCKS

for param in model.backbone.embeddings.parameters():
    param.requires_grad = False
for i in range(first_frozen_block):
    for param in model.backbone.encoder.layer[i].parameters():
        param.requires_grad = False

model.backbone.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print("Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)")

backbone_frozen = sum(p.numel() for p in model.backbone.parameters() if not p.requires_grad)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
head_trainable = sum(
    p.numel() for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
)
total = sum(p.numel() for p in model.parameters())

print(f"Backbone frozen:    {backbone_frozen:>12,} params")
print(f"Backbone trainable: {backbone_trainable:>12,} params (blocks {first_frozen_block}-23 + layernorm)")
print(f"Head trainable:     {head_trainable:>12,} params")
print(f"Total trainable:    {backbone_trainable + head_trainable:>12,} / {total:,} ({(backbone_trainable + head_trainable) / total * 100:.1f}%)")

model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"\nLogits shape: {out.logits.shape}")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()
print("Forward pass OK")

2026-03-18 22:02:57.240765: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773871377.256701    2673 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773871377.261609    2673 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773871377.275561    2673 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773871377.275573    2673 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773871377.275576    2673 computation_placer.cc:177] computation placer alr

[2026-03-18 22:03:03,788] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: No such file or directory
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)
Backbone frozen:     253,973,504 params
Backbone trainable:   50,395,136 params (blocks 20-23 + layernorm)
Head trainable:       36,720,660 params
Total trainable:      87,115,796 / 341,089,300 (25.5%)

Logits shape: torch.Size([1, 10, 518, 518])
Forward pass OK


In [0]:
# Cell 5 — Train with differential learning rate (200 epochs)

import torch
from datetime import datetime
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer

run_name = f"human-bigbrain-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}"
mlflow.start_run(run_name=run_name)
mlflow.log_params(HYPERPARAMS)

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=HEAD_LR,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    ddp_find_unused_parameters=True,
    dataloader_drop_last=True,
)

backbone_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=HYPERPARAMS["weight_decay"],
)

num_training_steps = (len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * HYPERPARAMS["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Run: {run_name}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Total steps: {num_training_steps} | Warmup: {num_warmup_steps}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()

2026/03/18 22:03:14 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/18 22:03:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Run: human-bigbrain-10class-20260318-2203
Epochs: 200
Total steps: 23400 | Warmup: 2340


Epoch,Training Loss,Validation Loss,Mean Iou,Overall Accuracy,Iou Class 0,Iou Class 1,Iou Class 2,Iou Class 3,Iou Class 4,Iou Class 5,Iou Class 6,Iou Class 7,Iou Class 8,Iou Class 9
1,2.001800,1.864731,0.274922,0.716612,0.933437,0.207621,0.446482,0.524216,0.226622,0.000043,0.009462,0.000000,0.000122,0.401218
2,1.266900,1.231313,0.390146,0.786545,0.934423,0.427888,0.547372,0.654931,0.201105,0.000000,0.640395,0.000000,0.000000,0.495347
3,0.955800,0.941563,0.428064,0.825687,0.946650,0.498201,0.598455,0.753020,0.237695,0.000000,0.725121,0.000000,0.000000,0.521502
4,0.774500,0.776396,0.450330,0.843340,0.956322,0.543143,0.630059,0.786272,0.311635,0.000000,0.751209,0.000000,0.000000,0.524662
5,0.680600,0.687268,0.461027,0.853852,0.960916,0.569700,0.671269,0.799179,0.299713,0.000056,0.770426,0.000000,0.001905,0.537106
6,0.607800,0.622316,0.468747,0.859190,0.963095,0.577449,0.677192,0.814770,0.335672,0.000902,0.781250,0.000000,0.003923,0.533215
7,0.571800,0.579038,0.476619,0.863331,0.963733,0.597202,0.690020,0.821218,0.348642,0.003088,0.790669,0.000000,0.012729,0.538894
8,0.522500,0.549601,0.480781,0.863882,0.966997,0.601963,0.694005,0.812397,0.364554,0.009035,0.796787,0.000000,0.026200,0.535876
9,0.504700,0.517100,0.482439,0.870917,0.970320,0.616139,0.703471,0.832729,0.325004,0.018259,0.786866,0.000000,0.022308,0.549291
10,0.475900,0.497847,0.488769,0.872913,0.970002,0.621478,0.706793,0.837940,0.347293,0.022304,0.801483,0.000000,0.029766,0.550629


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:r

TrainOutput(global_step=23400, training_loss=0.3588755931609716, metrics={'train_runtime': 4494.9192, 'train_samples_per_second': 20.824, 'train_steps_per_second': 5.206, 'total_flos': 1.5419701108318142e+20, 'train_loss': 0.3588755931609716, 'epoch': 200.0})

In [0]:
# Cell 6 — Evaluation: center-crop + sliding window

import numpy as np
from tqdm.auto import tqdm

# --- Center-crop evaluation ---
eval_results = trainer.evaluate()
print("Center-crop evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

current_miou = eval_results.get('eval_mean_iou', 0.0)
current_acc = eval_results.get('eval_overall_accuracy', 0.0)

mlflow.log_metrics({
    "cc_mean_iou": current_miou,
    "cc_overall_accuracy": current_acc,
})

# Per-class IoU
print(f"\nPer-class IoU:")
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    if not np.isnan(iou):
        print(f"  {cls:2d} ({name:25s}): {iou:.4f}")

# --- Sliding window evaluation ---
# BigBrain slices are grayscale, same as mouse — use grayscale normalization
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
STRIDE = CROP_SIZE // 2


def normalize_tile(tile):
    """uint8 grayscale (H, W) -> float32 tensor (1, 3, H, W)."""
    img = tile.astype(np.float32) / 255.0
    img_3ch = np.stack([img, img, img], axis=0)
    for c in range(3):
        img_3ch[c] = (img_3ch[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img_3ch).unsqueeze(0)


def sliding_window_predict(model, image, num_labels, crop_size, stride, device):
    """Predict full-resolution segmentation using overlapping tiles.

    Args:
        image: uint8 grayscale (H, W)
        Returns: int64 prediction map (H, W)
    """
    h, w = image.shape
    pad_h = max(0, crop_size - h)
    pad_w = max(0, crop_size - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image,
                       ((0, pad_h), (0, pad_w)),
                       mode='constant', constant_values=0)
    ph, pw = image.shape

    logit_sum = np.zeros((num_labels, ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)

    y_starts = list(range(0, ph - crop_size + 1, stride))
    x_starts = list(range(0, pw - crop_size + 1, stride))
    if y_starts[-1] + crop_size < ph:
        y_starts.append(ph - crop_size)
    if x_starts[-1] + crop_size < pw:
        x_starts.append(pw - crop_size)

    use_cuda = device.type == "cuda"
    model.eval()
    for y in y_starts:
        for x in x_starts:
            tile = image[y:y + crop_size, x:x + crop_size]
            pixel_values = normalize_tile(tile).to(device)
            with torch.no_grad():
                if use_cuda:
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        logits = model(pixel_values=pixel_values).logits
                else:
                    logits = model(pixel_values=pixel_values).logits
            tile_logits = logits.squeeze(0).float().cpu().numpy()
            logit_sum[:, y:y + crop_size, x:x + crop_size] += tile_logits
            count_map[y:y + crop_size, x:x + crop_size] += 1.0

    count_map = np.maximum(count_map, 1.0)
    avg_logits = logit_sum / count_map[np.newaxis, :, :]
    pred = avg_logits.argmax(axis=0).astype(np.int64)
    return pred[:h, :w]


# Run sliding window eval on all val slices
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

val_indices = splits["val"]
all_sw_preds = []
all_sw_labels = []

print(f"\nSliding window eval: {len(val_indices)} slices, crop={CROP_SIZE}, stride={STRIDE}")

for idx in tqdm(val_indices, desc="Sliding window eval"):
    img, annot = slicer.get_slice(idx, axis=0)
    class_mask = slicer._remap_mask(annot, mapping)

    pred = sliding_window_predict(
        model, img, NUM_LABELS, CROP_SIZE, STRIDE, device,
    )
    h, w = pred.shape
    class_mask = class_mask[:h, :w]

    all_sw_preds.append(pred.ravel())
    all_sw_labels.append(class_mask.ravel())

preds_flat = np.concatenate(all_sw_preds)
labels_flat = np.concatenate(all_sw_labels)

sw_accuracy = float((preds_flat == labels_flat).sum()) / len(labels_flat)

sw_class_ious = {}
for cls in range(NUM_LABELS):
    label_mask = labels_flat == cls
    if label_mask.sum() == 0:
        continue
    pred_mask = preds_flat == cls
    intersection = (pred_mask & label_mask).sum()
    union = (pred_mask | label_mask).sum()
    sw_class_ious[cls] = float(intersection) / float(union) if union > 0 else 0.0

sw_miou = float(np.mean(list(sw_class_ious.values()))) if sw_class_ious else 0.0
sw_valid = len(sw_class_ious)

print(f"\n{'='*65}")
print(f"EVALUATION RESULTS")
print(f"{'='*65}")
print(f"  Center-crop mIoU:        {current_miou:.1%}")
print(f"  Center-crop accuracy:    {current_acc:.1%}")
print(f"  Sliding window mIoU:     {sw_miou:.1%}")
print(f"  Sliding window accuracy: {sw_accuracy:.1%}")
print(f"  SW valid classes:        {sw_valid}")
print(f"  mIoU delta (SW - CC):    {sw_miou - current_miou:+.1%}")

print(f"\nPer-class sliding window IoU:")
for cls in range(NUM_LABELS):
    if cls in sw_class_ious:
        name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
        print(f"  {cls:2d} ({name:25s}): {sw_class_ious[cls]:.4f}")

mlflow.log_metrics({
    "sw_mean_iou": sw_miou,
    "sw_overall_accuracy": sw_accuracy,
    "sw_valid_classes": sw_valid,
    "sw_vs_cc_miou_delta": sw_miou - current_miou,
})

model = model.cpu()
torch.cuda.empty_cache()

Center-crop evaluation results:
  epoch: 200.0000
  eval_loss: 0.3427
  eval_mean_iou: 0.6061
  eval_overall_accuracy: 0.9001
  eval_runtime: 2.6301
  eval_samples_per_second: 22.4330
  eval_steps_per_second: 11.4070

Per-class IoU:
   0 (Background               ): 0.9864
   1 (Gray Matter              ): 0.7039
   2 (White Matter             ): 0.7754
   3 (Cerebrospinal Fluid      ): 0.8861
   4 (Meninges                 ): 0.3605
   5 (Blood Vessels            ): 0.1577
   6 (Bone/Skull               ): 0.8388
   7 (Muscle                   ): 0.7460
   8 (Artifact                 ): 0.0380
   9 (Other/Unknown            ): 0.5685

Sliding window eval: 59 slices, crop=518, stride=259


Sliding window eval:   0%|          | 0/59 [00:00<?, ?it/s]


EVALUATION RESULTS
  Center-crop mIoU:        60.6%
  Center-crop accuracy:    90.0%
  Sliding window mIoU:     61.2%
  Sliding window accuracy: 93.7%
  SW valid classes:        10
  mIoU delta (SW - CC):    +0.6%

Per-class sliding window IoU:
   0 (Background               ): 0.9941
   1 (Gray Matter              ): 0.7150
   2 (White Matter             ): 0.7849
   3 (Cerebrospinal Fluid      ): 0.8866
   4 (Meninges                 ): 0.3644
   5 (Blood Vessels            ): 0.1571
   6 (Bone/Skull               ): 0.8416
   7 (Muscle                   ): 0.7770
   8 (Artifact                 ): 0.0340
   9 (Other/Unknown            ): 0.5625


In [0]:
# Cell 7 — Save final model + log to MLflow + close run

import os
from transformers import AutoImageProcessor

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# 1. Set id2label/label2id
model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in enumerate(class_names)}
print(f"Set id2label/label2id: {NUM_LABELS} classes")

# 2. Save model to DBFS
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# 3. Save image processor
processor = AutoImageProcessor.from_pretrained(model_path)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"Image processor saved to: {FINAL_MODEL_DIR}")

# 4. Log to MLflow — raw file artifacts
mlflow.log_artifacts(FINAL_MODEL_DIR, artifact_path="model")
print("Model artifacts logged to MLflow under 'model/'")

# 5. Close MLflow run
mlflow.end_run()
print(f"\nMLflow run closed. Final model: {FINAL_MODEL_DIR}")

Set id2label/label2id: 10 classes


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model saved to: /dbfs/FileStore/allen_brain_data/models/human-bigbrain
Image processor saved to: /dbfs/FileStore/allen_brain_data/models/human-bigbrain


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

Uploading /dbfs/FileStore/allen_brain_data/models/human-bigbrain/model.safetensors:   0%|          | 0.00/1.27…

Model artifacts logged to MLflow under 'model/'


2026/03/18 23:18:32 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/18 23:18:32 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!



MLflow run closed. Final model: /dbfs/FileStore/allen_brain_data/models/human-bigbrain
